In [2]:
import sys
sys.path.append("..")

import time, os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, precision_score
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from src.pipeline import buat_pipeline, muat_data, pilih_skenario

MODEL = {
    "Baseline": (DummyClassifier(strategy="most_frequent"), {}),
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced", random_state=42),
                      {"model__max_depth": [3, 5, 8], "model__min_samples_leaf": [5, 20]}),
    "Random Forest": (RandomForestClassifier(class_weight="balanced", n_estimators=100,
                                             random_state=42, n_jobs=-1),
                      {"model__max_depth": [10, None], "model__min_samples_leaf": [5]}),
    "SVM": (SVC(class_weight="balanced", random_state=42),
            {"model__C": [0.1, 1, 10], "model__gamma": ["scale"]}),
    "kNN": (KNeighborsClassifier(), {"model__n_neighbors": [5, 15, 31]}),
}

N_REPEATS = 1   # percobaan pertama; nanti boleh dinaikkan jadi 2
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=42)
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
SKOR = {"f1": "f1", "recall": "recall",
        "precision": make_scorer(precision_score, zero_division=0),
        "roc_auc": "roc_auc"}

In [3]:
X_train, y_train = muat_data("../data/processed/train.csv")

hasil, baris = {}, []
for dengan in (False, True):
    skenario = "dengan akademik" if dengan else "tanpa akademik"
    Xs = pilih_skenario(X_train, dengan)
    for nama, (model, grid) in MODEL.items():
        t = time.time()
        pipe = buat_pipeline(model, Xs)
        est = GridSearchCV(pipe, grid, cv=inner, scoring="f1", n_jobs=-1) if grid else pipe
        r = cross_validate(est, Xs, y_train, cv=outer, scoring=SKOR, n_jobs=1)
        hasil[(skenario, nama)] = r
        baris.append({"skenario": skenario, "model": nama,
                      "F1": r["test_f1"].mean(), "F1_std": r["test_f1"].std(),
                      "recall": r["test_recall"].mean(), "precision": r["test_precision"].mean(),
                      "ROC_AUC": r["test_roc_auc"].mean()})
        print(f"{skenario} | {nama:14s} | F1 {r['test_f1'].mean():.3f} | {time.time()-t:.0f} dtk", flush=True)

tabel = pd.DataFrame(baris).round(3)
tabel

tanpa akademik | Baseline       | F1 0.000 | 0 dtk
tanpa akademik | Decision Tree  | F1 0.599 | 8 dtk
tanpa akademik | Random Forest  | F1 0.634 | 6 dtk
tanpa akademik | SVM            | F1 0.644 | 9 dtk
tanpa akademik | kNN            | F1 0.469 | 2 dtk
dengan akademik | Baseline       | F1 0.000 | 0 dtk
dengan akademik | Decision Tree  | F1 0.748 | 1 dtk
dengan akademik | Random Forest  | F1 0.769 | 5 dtk
dengan akademik | SVM            | F1 0.783 | 7 dtk
dengan akademik | kNN            | F1 0.682 | 2 dtk


,skenario,model,F1,F1_std,recall,precision,ROC_AUC
0,tanpa akademik,Baseline,0.000,0.000,0.000,0.000,0.500
1,tanpa akademik,Decision Tree,0.599,0.025,0.620,0.584,0.763
2,tanpa akademik,Random Forest,0.634,0.014,0.679,0.595,0.813
3,tanpa akademik,SVM,0.644,0.006,0.657,0.632,0.818
4,tanpa akademik,kNN,0.469,0.020,0.373,0.632,0.698
5,dengan akademik,Baseline,0.000,0.000,0.000,0.000,0.500
6,dengan akademik,Decision Tree,0.748,0.018,0.790,0.710,0.870
7,dengan akademik,Random Forest,0.769,0.020,0.793,0.747,0.907
8,dengan akademik,SVM,0.783,0.010,0.781,0.785,0.916
9,dengan akademik,kNN,0.682,0.025,0.581,0.827,0.839


In [4]:
def selang_kepercayaan(skor, k=5, alpha=0.05):
    J = len(skor)
    se = np.sqrt((1 / J + 1 / (k - 1)) * np.var(skor, ddof=1))
    t = stats.t.ppf(1 - alpha / 2, J - 1)
    return skor.mean() - t * se, skor.mean() + t * se

ci = []
for (skenario, nama), r in hasil.items():
    lo, hi = selang_kepercayaan(r["test_f1"])
    ci.append({"skenario": skenario, "model": nama, "F1_CI_bawah": lo, "F1_CI_atas": hi})

tabel_akhir = tabel.merge(pd.DataFrame(ci), on=["skenario", "model"]).round(3)
os.makedirs("../results", exist_ok=True)
tabel_akhir.to_csv("../results/hasil_cv.csv", index=False)
tabel_akhir

,skenario,model,F1,F1_std,recall,precision,ROC_AUC,F1_CI_bawah,F1_CI_atas
0,tanpa akademik,Baseline,0.000,0.000,0.000,0.000,0.500,0.000,0.000
1,tanpa akademik,Decision Tree,0.599,0.025,0.620,0.584,0.763,0.546,0.652
2,tanpa akademik,Random Forest,0.634,0.014,0.679,0.595,0.813,0.604,0.664
3,tanpa akademik,SVM,0.644,0.006,0.657,0.632,0.818,0.632,0.656
4,tanpa akademik,kNN,0.469,0.020,0.373,0.632,0.698,0.427,0.510
5,dengan akademik,Baseline,0.000,0.000,0.000,0.000,0.500,0.000,0.000
6,dengan akademik,Decision Tree,0.748,0.018,0.790,0.710,0.870,0.710,0.785
7,dengan akademik,Random Forest,0.769,0.020,0.793,0.747,0.907,0.728,0.810
8,dengan akademik,SVM,0.783,0.010,0.781,0.785,0.916,0.762,0.803
9,dengan akademik,kNN,0.682,0.025,0.581,0.827,0.839,0.630,0.735


In [5]:
MODEL_FINAL = "Random Forest"   # sesuai aturan yang sudah ditulis

In [6]:
import json
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, confusion_matrix
from src.pipeline import bersihkan_nama_kolom, TARGET

test = bersihkan_nama_kolom(pd.read_csv("../data/processed/test.csv"))
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

model, grid = MODEL[MODEL_FINAL]
uji, prediksi = {}, {}
for dengan in (False, True):
    skenario = "dengan akademik" if dengan else "tanpa akademik"
    Xs, Xt = pilih_skenario(X_train, dengan), pilih_skenario(X_test, dengan)
    gs = GridSearchCV(buat_pipeline(model, Xs), grid,
                      cv=StratifiedKFold(5, shuffle=True, random_state=42),
                      scoring="f1", n_jobs=-1).fit(Xs, y_train)
    pred = gs.predict(Xt)
    skor = gs.decision_function(Xt) if hasattr(gs, "decision_function") else gs.predict_proba(Xt)[:, 1]
    prediksi[skenario] = pred
    uji[skenario] = {
        "model": MODEL_FINAL, "param_terbaik": {k: str(v) for k, v in gs.best_params_.items()},
        "F1": round(f1_score(y_test, pred), 3), "recall": round(recall_score(y_test, pred), 3),
        "precision": round(precision_score(y_test, pred), 3), "ROC_AUC": round(roc_auc_score(y_test, skor), 3),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }

os.makedirs("../results", exist_ok=True)
with open("../results/hasil_uji_sekali.json", "w") as f:
    json.dump(uji, f, indent=2)
pd.DataFrame(uji).T

,model,param_terbaik,F1,recall,precision,ROC_AUC,confusion_matrix
tanpa akademik,Random Forest,"{'model__max_depth': '10', 'model__min_samples...",0.65,0.701,0.607,0.816,"[[472, 129], [85, 199]]"
dengan akademik,Random Forest,"{'model__max_depth': '10', 'model__min_samples...",0.795,0.806,0.784,0.921,"[[538, 63], [55, 229]]"


In [7]:
d = X_test.copy()
d["y"], d["pred"] = y_test.values, prediksi["tanpa akademik"]
d["hasil"] = np.select(
    [(d.y == 1) & (d.pred == 1), (d.y == 1) & (d.pred == 0), (d.y == 0) & (d.pred == 1)],
    ["TP (dropout, tertangkap)", "FN (dropout, terlewat)", "FP (salah alarm)"], "TN (benar tidak dropout)")

print(d["hasil"].value_counts())
d.groupby("hasil")[["Tuition fees up to date", "Scholarship holder", "Debtor",
                    "Age at enrollment", "Admission grade"]].mean().round(2)

hasil
TN (benar tidak dropout)    472
TP (dropout, tertangkap)    199
FP (salah alarm)            129
FN (dropout, terlewat)       85
Name: count, dtype: int64


,Tuition fees up to date,Scholarship holder,Debtor,Age at enrollment,Admission grade
hasil,,,,,
"FN (dropout, terlewat)",0.99,0.22,0.01,20.27,126.78
FP (salah alarm),0.90,0.08,0.14,29.60,127.84
TN (benar tidak dropout),0.99,0.41,0.04,19.82,128.28
"TP (dropout, tertangkap)",0.51,0.05,0.26,29.89,123.28
